# UrbanFlow - reproducible engine comparison

Measures identical single-column Parquet reads and pickup-zone counts using Pandas and Spark SQL. Three repetitions per engine; alternating engine order; exact count equality checked by the reusable module. Startup and sample preparation are excluded, caches are not cleared, input is coalesced to one file, and these are not full-ETL or distributed-cluster benchmarks. This writes a new experimental manifest rather than changing the original published run.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from datetime import UTC, datetime
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/UrbanFlow")
ORIGINAL_RUN = DRIVE / "runs/yellow-2025-q1-v1"
# This pinned archive is the exact reusable code used for the reported Q1 run.
BUNDLE = ORIGINAL_RUN / "executed-source.zip"
EXPECTED_SHA256 = "b9aeb8ea22e05f9b3d776c90dd3d4ac256fc6bb86f5da5f2554e8005d5b95aae"
if hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise ValueError("Source bundle checksum mismatch")
REPO = Path("/content/urbanflow-notebook-source").resolve()
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for member in archive.infolist():
        if not (REPO / member.filename).resolve().is_relative_to(REPO):
            raise ValueError("Unsafe archive member")
    archive.extractall(REPO)
# Explicit dependencies also support the archived package before geometry/benchmark extras.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(REPO / "pipelines"),
        "pyspark==3.5.9",
        "pandas==2.2.3",
        "pyarrow>=19,<24",
        "psutil>=6,<8",
        "pyshp>=2.3,<3",
        "pyproj>=3.6,<4",
    ],
    check=True,
)
sys.path.insert(0, str(REPO / "pipelines/src"))
RAW = Path("/content/urbanflow-data/raw")
CONFIG = json.loads((REPO / "configs/yellow-2025-q1.json").read_text())
print("Verified source:", EXPECTED_SHA256)

In [ ]:
from urbanflow_etl.ingestion.download import download_dataset

source_manifest = download_dataset(CONFIG, RAW)
INPUTS = [str(RAW / item["name"]) for item in CONFIG["files"] if item["kind"] == "parquet"]
print("Validated source inputs:", INPUTS)

In [ ]:
SERVING = Path("/content/urbanflow-benchmark-results/serving")
SERVING.mkdir(parents=True, exist_ok=True)
archive_path = ORIGINAL_RUN / "serving.zip"
with archive_path.open("rb") as stream:
    digest = hashlib.file_digest(stream, "sha256").hexdigest()
if digest != "95a1bb3d5d220d17afaf0b30268c36321d651ee1d4ba768951017a674879646b":
    raise ValueError("Original serving archive checksum mismatch")
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        if not (SERVING / member.filename).resolve().is_relative_to(SERVING.resolve()):
            raise ValueError("Unsafe archive member")
    archive.extractall(SERVING)
manifest_path = SERVING / "manifest.json"
manifest = json.loads(manifest_path.read_text())
RUN_ID = "yellow-2025-q1-benchmark-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
manifest["run_id"] = RUN_ID
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

In [ ]:
# Set driver memory before creating any Spark session. Restart runtime to change it later.
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 4g pyspark-shell"
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("UrbanFlow notebook")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
from urbanflow_etl.benchmarking.job import run as benchmark

try:
    experiments = benchmark(spark, INPUTS, SERVING, sizes=(10_000, 100_000, 1_000_000), repeats=3)
finally:
    spark.stop()
import pandas as pd

pd.DataFrame(experiments)[
    [
        "engine",
        "input_rows",
        "input_bytes",
        "elapsed_ms",
        "min_ms",
        "max_ms",
        "correctness_verified",
    ]
]

In [ ]:
destination = DRIVE / "runs" / RUN_ID
destination.mkdir(parents=True, exist_ok=False)
for filename in ["benchmarks.jsonl", "manifest.json"]:
    source = SERVING / filename
    target = destination / filename
    shutil.copyfile(source, target)
    if hashlib.sha256(source.read_bytes()).digest() != hashlib.sha256(target.read_bytes()).digest():
        raise ValueError("Drive checksum mismatch: " + filename)
print("Verified experiment evidence:", destination)

This experiment folder contains benchmark evidence, not a new complete serving archive or a published database version. Compare medians alongside all samples and the machine specification. Do not assume Spark must win; Pandas was faster on this task in the original Q1 run.